## 05 - Mesure de Risque (VaR, Expected Shortfall) & Stress Testing

Analyse globale des risques extrêmes d'un portefeuille via la Value-at-Risk, l'Expected Shortfall et la simulation de scénarios de crise boursière.

In [ ]:
import sys, os
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('../..'))

from src.portfolio.port import Portfolio, Position
from src.risk.var import ValueAtRisk
from src.risk.expected_shortfall import ExpectedShortfall
from src.risk.stress_testing import StressTestEngine

# Setup Portefeuille
port = Portfolio()
port.add_position(Position(asset_type="stock", quantity=1000, S=100.0))
port.add_position(Position(asset_type="option", quantity=50, S=100.0, K=100.0, T=0.25, r=0.03, sigma=0.20, option_type="call"))

val_total = port.total_value()
daily_vol = 0.015 # 1.5% vol quotidienne

# 1. Calculs de VaR (95% sur 1 jour)
var_param = ValueAtRisk.parametric_var(val_total, vol_daily=daily_vol, confidence_level=0.95)
var_mc = ValueAtRisk.monte_carlo_var(val_total, mu_daily=0.0, vol_daily=daily_vol, confidence_level=0.95)

# 2. Simulation de rendements pour VaR Historique & Expected Shortfall
np.random.seed(42)
returns_hist = np.random.normal(0, daily_vol, 1000)
var_hist = ValueAtRisk.historical_var(returns_hist, val_total, confidence_level=0.95)
es_hist = ExpectedShortfall.historical_es(returns_hist, val_total, confidence_level=0.95)

print(f"--- RAPPORT DE RISQUE (Niveau de Confiance 95% - Horizon 1 Jour) ---")
print(f"Valeur Portefeuille   : {val_total:,.2f} €")
print(f"VaR Paramétrique     : {var_param:,.2f} €")
print(f"VaR Monte-Carlo      : {var_mc:,.2f} €")
print(f"VaR Historique       : {var_hist:,.2f} €")
print(f"Expected Shortfall   : {es_hist:,.2f} €")

ModuleNotFoundError: No module named 'src.portfolio.portfolio'

In [ ]:
# 3. Stress Testing (Scénarios de Crise)
stress_engine = StressTestEngine(port)

scenarios = [
    ("Marché Baissier (-10%)", -0.10, 0.10),
    ("Crash Boursier (-20% Spot, +30% Vol)", -0.20, 0.30),
    ("Choc de Volatilité Pure (+50% Vol)", 0.00, 0.50)
]

print("\n--- RÉSULTATS DU STRESS TESTING ---")
for name, s_shock, v_shock in scenarios:
    res = stress_engine.run_scenario(s_shock, v_shock)
    print(f"\nScénario : {name}")
    print(f"  P&L Impact     : {res['pnl_impact']:,.2f} €")
    print(f"  Variation (%)  : {res['pnl_percentage']:.2f}%")
